# MediterraneanWillForge Pipeline Data Report

**Project:** MediterraneanWillForge  
**Stack:** Open-Meteo + OpenAQ v3 + WAQI → Delta Lake (Backblaze B2) → Gold marts + Anomaly Detection  
**Observability:** Grafana Cloud (`mohamedwillforge.grafana.net`)  

This notebook reads the hosted Backblaze B2 Gold tables and documents the committed report snapshot: geographic and temporal coverage, pollutant concentration levels, WHO guideline exceedance rates, wildfire risk distribution, anomaly detection results, and data source coverage.

Published artifacts are generated from the retained real-source lakehouse. Test fixtures are never used by report generation.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from deltalake import DeltaTable
from dotenv import load_dotenv

from data.reporting.analytics import (
    anomaly_daily_rates,
    filter_anomaly_model_sources,
    filter_report_countries,
    latest_reporting_dates,
    reporting_dates,
    write_readiness_diagnostics,
)

warnings.filterwarnings("ignore")

# Load B2 credentials from .env (MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY)
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))
load_dotenv()  # fallback: current dir .env

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f8f8",
    "axes.grid": True,
    "grid.color": "white",
    "grid.linewidth": 1.2,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = ["#2196F3", "#FF5722", "#4CAF50", "#FFC107", "#9C27B0", "#00BCD4"]
MAX_DATE_TICKS = 12


def date_tick_positions(count: int, max_ticks: int = MAX_DATE_TICKS) -> list[int]:
    if count <= 0:
        return []
    stride = max(1, int(np.ceil(count / max_ticks)))
    ticks = list(range(0, count, stride))
    if ticks[-1] != count - 1:
        ticks.append(count - 1)
    return ticks


print("Environment ready.")


In [ ]:
def storage_options() -> dict:
    endpoint = os.environ["MINIO_ENDPOINT"]
    return {
        "AWS_ENDPOINT_URL": endpoint,
        "AWS_ACCESS_KEY_ID": os.environ["MINIO_ACCESS_KEY"],
        "AWS_SECRET_ACCESS_KEY": os.environ["MINIO_SECRET_KEY"],
        "AWS_REGION": os.environ.get("AWS_REGION", "eu-central-003"),
        "AWS_ALLOW_HTTP": "true" if endpoint.startswith("http://") else "false",
        "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    }

GOLD = os.environ.get("MINIO_BUCKET_GOLD", "med-ops-mohamed-gold")
opts = storage_options()

def read_gold(table: str) -> pd.DataFrame:
    path = f"s3://{GOLD}/{table}"
    df = DeltaTable(path, storage_options=opts).to_pandas()
    if df.empty:
        raise RuntimeError(f"Gold table {table} is empty; refusing to publish report artifacts")
    print(f"  {table}: {len(df):,} rows")
    return df

print("Loading Gold tables from B2...")
summary   = filter_report_countries(read_gold("daily_country_summary"))
risk      = filter_report_countries(read_gold("wildfire_risk_index"))
anomalies = filter_anomaly_model_sources(
    filter_report_countries(read_gold("anomaly_alerts"))
)

READY_DATES = reporting_dates(summary)
DISPLAY_DATES = latest_reporting_dates(summary)
readiness = write_readiness_diagnostics(summary, anomalies, "reporting_readiness.csv")
print(f"Reporting-ready dates: {len(READY_DATES):,} of {summary['partition_date'].nunique() if not summary.empty else 0:,}")
print(f"Dense chart window: latest {len(DISPLAY_DATES):,} reporting-ready dates")
print("Latest date diagnostics:")
print(readiness.tail(5).to_string(index=False))
print("Done.")


---
## 1. Pipeline Coverage

How much data was collected, across which countries and dates.

In [ ]:
# ── Summary stats ──────────────────────────────────────────────────────────────
dates      = sorted(summary["partition_date"].unique())
countries  = sorted(summary["country_code"].dropna().unique())
sources    = sorted(summary["source"].dropna().unique())

print(f"Date range  : {dates[0]}  →  {dates[-1]}")
print(f"Dates total : {len(dates)}")
print(f"Countries   : {len(countries)}  —  {', '.join(countries)}")
print(f"Sources     : {', '.join(sources)}")
print(f"\nSummary rows    : {len(summary):,}")
print(f"Risk index rows : {len(risk):,}")
print(f"Anomaly rows    : {len(anomalies):,}")
total_stations = risk["station_id"].nunique() if not risk.empty else "N/A"
print(f"Unique stations : {total_stations}")

In [ ]:
# Station-day heatmap per country, limited to the latest reporting-ready dates.
public_summary = summary[summary["partition_date"].isin(DISPLAY_DATES)].copy()

pivot = (
    public_summary
    .groupby(["partition_date", "country_code"])["station_count"]
    .sum()
    .unstack("country_code", fill_value=0)
)
pivot.index = pd.to_datetime(pivot.index)

fig, ax = plt.subplots(figsize=(14, max(4, len(pivot.columns) * 0.45)))
im = ax.imshow(
    pivot.values.T,
    aspect="auto",
    cmap="YlOrRd",
    interpolation="nearest",
)
ax.set_yticks(range(len(pivot.columns)))
ax.set_yticklabels(pivot.columns, fontsize=9)
ticks = date_tick_positions(len(pivot.index))
ax.set_xticks(ticks)
ax.set_xticklabels(
    [pivot.index[t].strftime("%b %d") for t in ticks],
    rotation=60, ha="right", fontsize=7
)
plt.colorbar(im, ax=ax, label="Station-days")
ax.set_title(
    f"Station Coverage per Country - Latest {len(DISPLAY_DATES)} Stable Dates",
    fontsize=13,
    fontweight="bold",
    pad=12,
)
ax.set_facecolor("white")
plt.tight_layout()
plt.savefig("coverage_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 2. Pollutant Concentrations by Country

Mean PM2.5 and O3 across all collected dates, split by data source.

In [ ]:
# ── Mean PM2.5 by country × source ────────────────────────────────────────────
pm25_by_country = (
    summary
    .groupby(["country_code", "source"])["mean_pm2_5"]
    .mean()
    .round(2)
    .unstack("source", fill_value=np.nan)
    .sort_values(by=summary["source"].mode()[0], ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PM2.5
ax = axes[0]
x = np.arange(len(pm25_by_country))
w = 0.35
for i, (src, col) in enumerate(zip(pm25_by_country.columns, PALETTE)):
    vals = pm25_by_country[src].values
    bars = ax.barh(x + i * w - w/2, vals, height=w, label=src, color=col, alpha=0.85)
ax.axvline(15, color="red", linestyle="--", linewidth=1, label="WHO PM2.5 guideline (15 µg/m³)")
ax.set_yticks(x)
ax.set_yticklabels(pm25_by_country.index)
ax.set_xlabel("Mean PM2.5 (µg/m³)")
ax.set_title("Mean PM2.5 by Country", fontweight="bold")
ax.legend(fontsize=8)

# O3
o3_by_country = (
    summary
    .groupby(["country_code", "source"])["mean_o3"]
    .mean()
    .round(2)
    .unstack("source", fill_value=np.nan)
    .reindex(pm25_by_country.index)
)
ax = axes[1]
for i, (src, col) in enumerate(zip(o3_by_country.columns, PALETTE)):
    vals = o3_by_country[src].values
    ax.barh(x + i * w - w/2, vals, height=w, label=src, color=col, alpha=0.85)
ax.axvline(100, color="red", linestyle="--", linewidth=1, label="WHO O3 guideline (100 µg/m³)")
ax.set_yticks(x)
ax.set_yticklabels(pm25_by_country.index)
ax.set_xlabel("Mean O3 (µg/m³)")
ax.set_title("Mean O3 by Country", fontweight="bold")
ax.legend(fontsize=8)

plt.suptitle("Air Pollutant Concentrations — All Dates Combined", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("pollutants_by_country.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3. WHO Guideline Exceedance

Percentage of station-days where concentrations exceeded WHO 2021 guidelines.

In [ ]:
who_cols = {
    "who_pm25_exceed_pct": "PM2.5",
    "who_pm10_exceed_pct": "PM10",
    "who_no2_exceed_pct":  "NO2",
    "who_o3_exceed_pct":   "O3",
}
who_cols = {k: v for k, v in who_cols.items() if k in summary.columns}

exceed = (
    summary
    .groupby("country_code")[list(who_cols.keys())]
    .mean()
    .round(1)
    .rename(columns=who_cols)
)
exceed = exceed.sort_values(by=list(who_cols.values())[0], ascending=True)

fig, ax = plt.subplots(figsize=(11, max(4, len(exceed) * 0.55)))
x = np.arange(len(exceed))
n = len(who_cols)
w = 0.18
for i, (col, color) in enumerate(zip(exceed.columns, PALETTE)):
    ax.barh(
        x + (i - n / 2 + 0.5) * w,
        exceed[col],
        height=w,
        label=col,
        color=color,
        alpha=0.85,
    )
ax.set_yticks(x)
ax.set_yticklabels(exceed.index)
ax.set_xlabel("% of station-days exceeding WHO guideline")
ax.set_title("WHO Guideline Exceedance Rate by Country", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title="Pollutant", loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("who_exceedance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nCountries with highest PM2.5 exceedance rate:")
print(exceed[["PM2.5"]].sort_values("PM2.5", ascending=False).head(5).to_string())

---
## 4. Wildfire Risk Index

Composite score (0–100) per station × day: 60% normalised O3 + 40% normalised PM2.5.  
Levels: **low** (<25) · **moderate** (25–50) · **high** (50–75) · **extreme** (>75)

In [ ]:
if risk.empty:
    print("No wildfire risk data available.")
else:
    level_order = ["low", "moderate", "high", "extreme"]
    level_colors = {"low": "#4CAF50", "moderate": "#FFC107", "high": "#FF5722", "extreme": "#B71C1C"}

    # ── Distribution of risk levels by country ─────────────────────────────────
    risk_counts = (
        risk
        .groupby(["country_code", "risk_level"])
        .size()
        .unstack("risk_level", fill_value=0)
        .reindex(columns=[l for l in level_order if l in risk["risk_level"].unique()], fill_value=0)
    )
    risk_pct = risk_counts.div(risk_counts.sum(axis=1), axis=0) * 100
    risk_pct = risk_pct.sort_values(by=[c for c in ["extreme", "high"] if c in risk_pct.columns], ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(risk_pct) * 0.55)))

    # Stacked bar: risk level distribution
    ax = axes[0]
    left = np.zeros(len(risk_pct))
    for level in risk_pct.columns:
        vals = risk_pct[level].values
        ax.barh(risk_pct.index, vals, left=left, label=level, color=level_colors.get(level, "grey"), alpha=0.9)
        left += vals
    ax.set_xlabel("% of station-days")
    ax.set_title("Risk Level Distribution by Country", fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.legend(title="Risk level", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

    # Scatter: O3 vs PM2.5 coloured by risk level
    ax = axes[1]
    for level in level_order:
        mask = risk["risk_level"] == level
        if mask.any():
            ax.scatter(
                risk.loc[mask, "pm2_5"],
                risk.loc[mask, "ozone"],
                c=level_colors[level],
                label=level,
                alpha=0.5,
                s=12,
            )
    ax.set_xlabel("PM2.5 (µg/m³)")
    ax.set_ylabel("O3 (µg/m³)")
    ax.set_title("PM2.5 vs O3 — Risk Level", fontweight="bold")
    ax.legend(title="Risk level", fontsize=8)

    plt.suptitle("Wildfire Risk Index", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("wildfire_risk.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nTop 10 highest-risk station-days:")
    cols = [c for c in ["partition_date", "country_code", "station_name", "risk_index", "risk_level", "pm2_5", "ozone"] if c in risk.columns]
    print(risk.nlargest(10, "risk_index")[cols].to_string(index=False))

---
## 5. Anomaly Detection

WAQI is excluded from anomaly detection because its `iaqi` pollutant fields are index values rather than raw concentrations. The model uses Open-Meteo and OpenAQ concentration-compatible rows only.

Isolation Forest trained on the full Silver history (PM2.5, O3, NO2).  
Contamination rate: 5%.

In [ ]:
public_anomalies = anomalies[anomalies["partition_date"].isin(DISPLAY_DATES)].copy()

if public_anomalies.empty:
    print("No anomaly data available for the latest reporting-ready dates.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Anomaly rate per date. Raw counts are misleading when latest coverage changes.
    ax = axes[0]
    anom_by_date = anomaly_daily_rates(public_anomalies)
    x = range(len(anom_by_date))
    ax.bar(x, anom_by_date["anomaly_rate_pct"], color="#FF5722", alpha=0.8)
    ticks = date_tick_positions(len(anom_by_date))
    ax.set_xticks(ticks)
    ax.set_xticklabels(
        [str(anom_by_date.iloc[t]["partition_date"])[-5:] for t in ticks],
        rotation=70, ha="right", fontsize=7
    )
    ax.set_ylabel("Anomaly rate")
    ax.set_title("Anomaly Rate per Date", fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())

    # Anomaly rate by country
    ax = axes[1]
    anom_by_country = (
        public_anomalies
        .groupby("country_code")["is_anomaly"]
        .agg(["sum", "count"])
        .rename(columns={"sum": "anomalies", "count": "total"})
    )
    anom_by_country["rate"] = (anom_by_country["anomalies"] / anom_by_country["total"] * 100).round(1)
    anom_by_country = anom_by_country.sort_values("rate", ascending=True)
    ax.barh(anom_by_country.index, anom_by_country["rate"], color="#9C27B0", alpha=0.8)
    ax.set_xlabel("Anomaly rate")
    ax.set_title("Anomaly Rate by Country", fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())

    # Anomaly score distribution
    ax = axes[2]
    normal = public_anomalies.loc[public_anomalies["is_anomaly"] == 0, "anomaly_score"]
    flagged = public_anomalies.loc[public_anomalies["is_anomaly"] == 1, "anomaly_score"]
    ax.hist(normal, bins=30, color="#4CAF50", alpha=0.7, label=f"Normal ({len(normal):,})")
    ax.hist(flagged, bins=15, color="#FF5722", alpha=0.85, label=f"Anomaly ({len(flagged):,})")
    ax.set_xlabel("Isolation Forest score (lower = more anomalous)")
    ax.set_ylabel("Count")
    ax.set_title("Score Distribution", fontweight="bold")
    ax.legend(fontsize=8)

    plt.suptitle(
        f"Anomaly Detection - Latest {len(DISPLAY_DATES)} Stable Dates",
        fontsize=13,
        fontweight="bold",
        y=1.01,
    )
    plt.tight_layout()
    plt.savefig("anomaly_detection.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nTop 10 most anomalous readings in chart window (lowest score = most deviant):")
    cols = [c for c in ["partition_date", "country_code", "station_name", "pm2_5", "ozone", "nitrogen_dioxide", "anomaly_score"] if c in public_anomalies.columns]
    print(public_anomalies[public_anomalies["is_anomaly"] == 1].nsmallest(10, "anomaly_score")[cols].to_string(index=False))


---
## 6. Top Anomaly of the Latest Date

The single most anomalous reading on the most recent reporting-ready date,
shown against that day's spread across every station. The marked line is the
flagged station; the histogram is every other station reporting that pollutant,
so the reason the Isolation Forest singled it out is visible rather than asserted.

A station with no sensor for a pollutant is labelled as having no reading — it is
not plotted as zero.


In [ ]:
# Spotlight the single most anomalous reading on the latest reporting-ready date.
# The three panels show that day's spread across every station, with the flagged
# station marked — so the reason it was flagged is visible, not just asserted.
TOP_DATE = max(DISPLAY_DATES) if len(DISPLAY_DATES) else None
day = anomalies[anomalies["partition_date"] == TOP_DATE].copy() if TOP_DATE else pd.DataFrame()

if day.empty:
    print("No anomaly data for the latest reporting-ready date.")
else:
    top = day.nsmallest(1, "anomaly_score").iloc[0]
    POLLUTANTS = [
        ("pm2_5", "PM2.5", "#2196F3"),
        ("ozone", "Ozone", "#4CAF50"),
        ("nitrogen_dioxide", "NO2", "#9C27B0"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
    for ax, (col, label, colour) in zip(axes, POLLUTANTS):
        values = pd.to_numeric(day[col], errors="coerce").dropna()
        station_value = pd.to_numeric(pd.Series([top.get(col)]), errors="coerce").iloc[0]

        if values.empty:
            ax.text(0.5, 0.5, f"No {label} readings\non this date",
                    ha="center", va="center", fontsize=10, color="#777")
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(label, fontweight="bold")
            continue

        ax.hist(values, bins=min(20, max(5, len(values) // 2)),
                color=colour, alpha=0.55, edgecolor="white")

        if pd.isna(station_value):
            # A station with no sensor for this pollutant reports nothing at all.
            ax.set_title(f"{label} — station has no reading", fontweight="bold", fontsize=10)
        else:
            pct = (values < station_value).mean() * 100
            ax.axvline(station_value, color="#FF5722", linewidth=2.5, zorder=5)
            # Label on whichever side of the line has room, so it never clips.
            lo, hi = ax.get_xlim()
            on_right = station_value > lo + 0.6 * (hi - lo)
            ax.annotate(
                f"{station_value:,.1f}\n{pct:.0f}th pct",
                xy=(station_value, ax.get_ylim()[1] * 0.92),
                xytext=(-7 if on_right else 7, 0), textcoords="offset points",
                ha="right" if on_right else "left",
                color="#FF5722", fontweight="bold", fontsize=9, va="top",
            )
            ax.set_title(f"{label} — median {values.median():,.1f}", fontweight="bold", fontsize=10)

        ax.set_xlabel(f"{label} (ug/m3)")
        ax.set_ylabel("Stations")

    flagged = "flagged anomaly" if int(top.get("is_anomaly", 0)) == 1 else "most deviant reading"
    plt.suptitle(
        f"Top Anomaly {TOP_DATE} — {top['station_name']} ({top['country_code']}, {top['source']}) "
        f"— score {top['anomaly_score']:.3f} — {flagged}",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.savefig("top_anomaly.png", dpi=150, bbox_inches="tight")
    plt.show()

    cols = [c for c in ["country_code", "station_name", "source", "pm2_5", "ozone",
                        "nitrogen_dioxide", "anomaly_score", "is_anomaly"] if c in day.columns]
    print(f"\nMost anomalous readings on {TOP_DATE} (lowest score = most deviant):")
    print(day.nsmallest(5, "anomaly_score")[cols].to_string(index=False))


---
## 7. AI Daily Brief

Written by Claude from this run's Gold layer. The anomaly fact-check is given the
flagged reading plus that day's distribution and searches the web for a real-world
explanation — a wildfire, dust intrusion, heatwave or traffic event. It is instructed
to report an absence of evidence as an absence of evidence; an uncorroborated anomaly
is the normal case, not a prompt to invent a cause.

This section is generated text and is labelled as such. Every figure it cites comes
from the pipeline; anything it claims about the wider world carries its source link.


In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display

brief_path = Path("ai_brief.json")

if not brief_path.exists():
    print("No AI brief for this run (GEMINI_API_KEY unset, or the call failed).")
else:
    brief = json.loads(brief_path.read_text(encoding="utf-8"))
    lines = [
        f"*Generated by `{brief.get('model') or 'Gemini'}` from the "
        f"{brief.get('date', 'latest')} Gold layer. Figures come from the pipeline; "
        "the fact-check additionally searched the web and cites its sources.*",
        "",
    ]

    fc = brief.get("fact_check")
    if fc:
        lines += [
            "### Anomaly fact-check",
            "",
            f"**{fc['station']} ({fc['country_code']})** — {fc['verdict']}",
            "",
        ]
        if fc.get("sources"):
            lines.append("Sources consulted:")
            lines += [f"- [{s['title']}]({s['url']})" for s in fc["sources"]]
            lines.append("")
        elif fc.get("searched") is False:
            lines += ["*No web sources were returned for this reading.*", ""]

    briefings = brief.get("briefings") or []
    if briefings:
        lines += ["### Country briefings", ""]
        for entry in sorted(briefings, key=lambda b: b.get("country_code", "")):
            lines.append(f"- **{entry['country_code']}** — {entry['briefing']}")
        lines.append("")

    display(Markdown("\n".join(lines)))
    print(f"AI brief: {len(briefings)} country briefing(s), "
          f"fact-check={'yes' if fc else 'no'}, generated {brief.get('generated_at', '?')}")


---
## 8. Source Coverage by Data Source

Station-days contributed by each source (Open-Meteo, OpenAQ, WAQI) per country.
WAQI was added in v1.7.0 specifically to fill coverage gaps for LB and MA where OpenAQ has sparse station density.

In [ ]:
public_summary = summary[summary["partition_date"].isin(READY_DATES)].copy()

if public_summary.empty:
    print("No summary data available for reporting-ready dates.")
else:
    coverage = (
        public_summary
        .groupby(["country_code", "source"])["station_count"]
        .sum()
        .unstack("source", fill_value=0)
    )
    coverage = coverage.loc[coverage.sum(axis=1).sort_values().index]

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(coverage) * 0.55)))

    # Absolute station-days per source
    ax = axes[0]
    left = np.zeros(len(coverage))
    for src, color in zip(coverage.columns, PALETTE):
        vals = coverage[src].values
        ax.barh(coverage.index, vals, left=left, label=src, color=color, alpha=0.85)
        left += vals
    ax.set_xlabel("Station-days")
    ax.set_title("Station-Days per Country by Source", fontweight="bold")
    ax.legend(title="Source", fontsize=8)

    # Percentage share per source
    coverage_pct = coverage.div(coverage.sum(axis=1), axis=0) * 100
    ax = axes[1]
    left = np.zeros(len(coverage_pct))
    for src, color in zip(coverage_pct.columns, PALETTE):
        vals = coverage_pct[src].values
        ax.barh(coverage_pct.index, vals, left=left, label=src, color=color, alpha=0.85)
        left += vals
    ax.set_xlabel("% of station-days")
    ax.set_title("Source Share per Country (%)", fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax.legend(title="Source", fontsize=8)

    plt.suptitle(
        "Data Source Coverage - Station-Days per Country",
        fontsize=13, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    plt.savefig("source_coverage.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nStation-days per source per country:")
    print(coverage.to_string())


---
## 9. Pipeline Summary

What the pipeline has produced as of this report.

In [ ]:
n_anomalies = int(anomalies["is_anomaly"].sum()) if not anomalies.empty else 0
n_anom_rows = len(anomalies)
anom_rate = f"{n_anomalies / n_anom_rows:.1%}" if n_anom_rows else "N/A"
top_risk_country = (
    risk.groupby("country_code")["risk_index"].mean().idxmax()
    if not risk.empty else "N/A"
)
highest_exceed_country = (
    summary.groupby("country_code")["who_pm25_exceed_pct"].mean().idxmax()
    if "who_pm25_exceed_pct" in summary.columns else "N/A"
)

report = pd.DataFrame([
    ["Date range",             f"{dates[0]} → {dates[-1]}"],
    ["Total dates collected",  len(dates)],
    ["Countries covered",      f"{len(countries)}  ({', '.join(countries)})"],
    ["Data sources",           ", ".join(sources)],
    ["Unique stations",        risk["station_id"].nunique() if not risk.empty else "N/A"],
    ["Gold summary rows",      f"{len(summary):,}"],
    ["Gold risk index rows",   f"{len(risk):,}"],
    ["Gold anomaly rows",      f"{n_anom_rows:,}"],
    ["Anomalies flagged",      f"{n_anomalies} ({anom_rate})"],
    ["Highest avg risk country", top_risk_country],
    ["Highest PM2.5 exceedance", highest_exceed_country],
], columns=["Metric", "Value"])

print(report.to_string(index=False))

---
## Regenerate the Report

Install `requirements-report.txt`, then run from the repository root:

```bash
cd docs
jupyter nbconvert --to notebook --execute pipeline_report.ipynb --output pipeline_report
jupyter nbconvert --to html pipeline_report.ipynb --output pipeline_report
```